# RSM v0.2 — RIPE sanity demo\n\nThis notebook demonstrates the central correction in v0.2: **recursive stability is not truth**.\n

In [ ]:
from pathlib import Path\nimport sys\n\nroot = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()\nsys.path.insert(0, str(root / 'src'))\n\nfrom rsm import run_audit\nfrom rsm.metrics import sequence_distance, token_jaccard_distance\n\nmetrics = {\n    'sequence': sequence_distance,\n    'tokens': token_jaccard_distance,\n}\n

## 1. Stable falsehood\n\nIdentity preserves a false sentence perfectly. RIPE should call the representation C1 while making **no factual claim**.\n

In [ ]:
false_claim = 'The Sun orbits Earth once per day.'\nstable_falsehood = run_audit(false_claim, lambda x: x, metrics, steps=6)\nprint('class:', stable_falsehood.classification)\nprint('trajectory:', stable_falsehood.mean_scores)\nprint('external truth label: false (supplied separately, not inferred by RIPE)')\n

## 2. The old reversal demo, diagnosed correctly\n\nString reversal creates a two-cycle. The liar sentence is irrelevant to the effect. Full-trajectory reporting exposes the artifact.\n

In [ ]:
cycle = run_audit(\n    'This sentence is false.',\n    lambda x: x[::-1],\n    {'sequence': sequence_distance},\n    steps=8,\n)\nprint('class:', cycle.classification)\nprint('trajectory:', tuple(round(x, 3) for x in cycle.mean_scores))\nprint('cycle period:', cycle.cycle_periods[0])\n

## 3. Destructive recursion\n\nProgressive truncation destroys declared surface/token structure and should move into C3.\n

In [ ]:
def erode(text):\n    return text[: max(0, len(text) // 2)]\n\nloss = run_audit(\n    'recursive structure should survive declared transforms',\n    erode,\n    metrics,\n    steps=6,\n)\nprint('class:', loss.classification)\nprint('trajectory:', tuple(round(x, 3) for x in loss.mean_scores))\n